# Data Quality Audit — `raw_credit_applications.json`
### MSc Business Analytics | Data Ecosystems and Governance in Organisations

This notebook identifies, quantifies, and remediates data quality issues across four standard dimensions:
- **Completeness** — are all required values present?
- **Consistency** — are values stored in a uniform format?
- **Validity** — are values within logically acceptable ranges?
- **Accuracy** — do values conform to real-world format standards?

| | |
|---|---|
| Total records loaded | 502 |
| Final clean records | 500 |
| Records removed (duplicates) | 2 |
| Records flagged (SSN conflict) | 4 |

## Step 0 — Imports and Load Data
`pd.json_normalize()` flattens the nested JSON structure so that nested fields like `applicant_info.gender` become flat column names, making the data easy to work with as a table.

In [83]:
import pandas as pd
import json
import re

with open('raw_credit_applications.json', 'r') as f:
    raw_data = json.load(f)

rc = pd.json_normalize(raw_data)
total = len(rc)
print(f"Total records loaded: {total}")
print(f"Columns: {list(rc.columns)}")
rc.head(3)

Total records loaded: 502
Columns: ['_id', 'spending_behavior', 'processing_timestamp', 'applicant_info.full_name', 'applicant_info.email', 'applicant_info.ssn', 'applicant_info.ip_address', 'applicant_info.gender', 'applicant_info.date_of_birth', 'applicant_info.zip_code', 'financials.annual_income', 'financials.credit_history_months', 'financials.debt_to_income', 'financials.savings_balance', 'decision.loan_approved', 'decision.rejection_reason', 'loan_purpose', 'decision.interest_rate', 'decision.approved_amount', 'financials.annual_salary', 'notes']


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN


## Issue 1 — Duplicate Records
**Dimension: Uniqueness**

Checks for two types of duplication:
- **Duplicate `_id`**: the same application ID appears more than once — a clear data entry error
- **SSN conflicts**: the same SSN is assigned to two different people — an accuracy problem, not a true duplicate

> **Key decision:** Records with missing SSN are NOT treated as duplicates. A missing SSN is a completeness issue (Issue 3).
> Repeated names are NOT removed — one person can hold multiple loans, and two different people can share a name.

In [84]:
#Identify duplicate _id
print("Duplicate _id (keep=False shows ALL occurrences):")
print(rc[rc.duplicated(subset=['_id'], keep=False)]
      [['_id','applicant_info.full_name','applicant_info.ssn']]
      .sort_values('_id').to_string(index=False))

# Identify SSN 
mask_real_ssn = (rc['applicant_info.ssn'].notna() &
                 (rc['applicant_info.ssn'].astype(str) != 'None'))

ssn_name_counts = (rc[mask_real_ssn]
                   .groupby('applicant_info.ssn')['applicant_info.full_name']
                   .nunique())

conflicting_ssns = ssn_name_counts[ssn_name_counts > 1].index

print("\nSSN conflict records (same SSN, different people):")
print(rc[rc['applicant_info.ssn'].isin(conflicting_ssns)]
      [['_id','applicant_info.full_name','applicant_info.ssn',
        'applicant_info.date_of_birth']]
      .to_string(index=False))

# Remove duplicate keep first 
rc = rc.drop_duplicates(subset=['_id'], keep='first')

rc['ssn_conflict_flag'] = rc['applicant_info.ssn'].isin(conflicting_ssns)

print(f"\nRecords after dedup: {len(rc)}")
print(f"SSN conflict records flagged for review: {rc['ssn_conflict_flag'].sum()}")

Duplicate _id (keep=False shows ALL occurrences):
    _id applicant_info.full_name applicant_info.ssn
app_001         Stephanie Nguyen        427-90-1892
app_001         Stephanie Nguyen                NaN
app_042             Joseph Lopez        652-70-5530
app_042             Joseph Lopez        652-70-5530

SSN conflict records (same SSN, different people):
    _id applicant_info.full_name applicant_info.ssn applicant_info.date_of_birth
app_101             Sandra Smith        937-72-8731                   1997-03-23
app_088           Susan Martinez        780-24-9300                   1986-10-15
app_016              Gary Wilson        780-24-9300                   1959-12-11
app_234              Samuel Hill        937-72-8731                   1976/01/29

Records after dedup: 500
SSN conflict records flagged for review: 4


## Issue 2 — SSN Format Validation
**Dimension: Accuracy**

A valid US Social Security Number must follow the format `NNN-NN-NNNN` (3 digits, dash, 2 digits, dash, 4 digits).
We use a regular expression to check every non-null SSN against this pattern.

In [85]:
# SSN must follow format
ssn_pattern = r'^\d{3}-\d{2}-\d{4}$'

invalid_ssn = rc['applicant_info.ssn'].dropna().apply(
    lambda x: not bool(re.match(ssn_pattern, str(x).strip()))
)
n_bad_ssn = invalid_ssn.sum()
print(f'Malformed SSNs (wrong format): {n_bad_ssn}')
if n_bad_ssn > 0:
    print(rc.loc[invalid_ssn[invalid_ssn].index,
                 ['_id','applicant_info.full_name','applicant_info.ssn']].to_string(index=False))
else:
    print("All present SSNs follow the correct format.")

Malformed SSNs (wrong format): 0
All present SSNs follow the correct format.


## Issue 3 — Annual Salary vs Annual Income
**Dimension: Consistency**

5 records used the field name `annual_salary` instead of the standard `annual_income`.

`fillna()` fills the empty `annual_income` slots with values from `annual_salary`, then the redundant column is dropped.

In [86]:
# Identify schema drift
has_salary = 'financials.annual_salary' in rc.columns
n_salary = int(rc['financials.annual_salary'].notna().sum()) if has_salary else 0
print(f"Records using annual_salary instead of annual_income: {n_salary}")

if has_salary:
    print(rc[rc['financials.annual_salary'].notna()]
          [['_id','applicant_info.full_name',
            'financials.annual_income','financials.annual_salary']]
          .to_string(index=False))

if has_salary:
    rc['financials.annual_income'] = (rc['financials.annual_income']
                                      .fillna(rc['financials.annual_salary']))
    rc.drop(columns=['financials.annual_salary'], inplace=True)

print(f"annual_income missing after fix: {rc['financials.annual_income'].isna().sum()}")

Records using annual_salary instead of annual_income: 5
    _id applicant_info.full_name financials.annual_income  financials.annual_salary
app_436            Amanda Torres                      NaN                   45000.0
app_421             Donald Baker                      NaN                   46000.0
app_479             Sandra Jones                      NaN                   94000.0
app_463               Emma Clark                      NaN                   86000.0
app_449             Lisa Roberts                      NaN                   75000.0
annual_income missing after fix: 0


## Issue 4 — Missing / Incomplete Records
**Dimension: Completeness**

Missing data can bias model outcomes and reduce the reliability of credit decisions.

> **Important distinction:** `rejection_reason`, `interest_rate`, and `approved_amount` are missing **by design** 

`loan_purpose`: optional field → fill with `'not_specified'`
Empty email strings: replace with `NaN`

In [87]:
# Count missing values per column
missing = rc.isna().sum()
pct     = (missing / len(rc) * 100).round(2)
report  = pd.DataFrame({'missing_count': missing, 'missing_%': pct})
report  = report[report['missing_count'] > 0].sort_values('missing_%', ascending=False)
print(report.to_string())

total_cells   = rc.shape[0] * rc.shape[1]
total_missing = missing.sum()
print(f"\nTotal missing cells: {total_missing}/{total_cells} ({total_missing/total_cells*100:.1f}%)")

print("\nFields missing by reason:")
print("  decision.rejection_reason -> only exists when loan_approved = False")
print("  decision.interest_rate    -> only exists when loan_approved = True")
print("  decision.approved_amount  -> only exists when loan_approved = True")

n_lp = rc['loan_purpose'].isna().sum()
rc['loan_purpose'] = rc['loan_purpose'].fillna('not_specified')
print(f"\nloan_purpose: {n_lp} missing -> filled with 'not_specified'")

rc['applicant_info.email'] = rc['applicant_info.email'].replace('', pd.NA)
print("email: empty strings replaced with NaN")



                           missing_count  missing_%
notes                                500      100.0
loan_purpose                         450       90.0
processing_timestamp                 438       87.6
decision.rejection_reason            292       58.4
decision.interest_rate               208       41.6
decision.approved_amount             208       41.6
applicant_info.ip_address              4        0.8
applicant_info.ssn                     4        0.8

Total missing cells: 2104/10500 (20.0%)

Fields missing by reason:
  decision.rejection_reason -> only exists when loan_approved = False
  decision.interest_rate    -> only exists when loan_approved = True
  decision.approved_amount  -> only exists when loan_approved = True

loan_purpose: 450 missing -> filled with 'not_specified'
email: empty strings replaced with NaN


## Issue 5 — Inconsistent Data Types
**Dimension: Consistency**


`annual_income` appears as `int`, `str`, and `float` 

`pd.to_numeric(errors='coerce')` converts everything possible to float.


In [88]:
# Detect mixed types
print("Row-by-row types in annual_income BEFORE fix:")
print(rc['financials.annual_income'].dropna().apply(type).value_counts().to_string())

for col in ['financials.annual_income', 'financials.credit_history_months',
            'financials.debt_to_income', 'financials.savings_balance',
            'decision.interest_rate', 'decision.approved_amount']:
    if col in rc.columns:
        rc[col] = pd.to_numeric(rc[col], errors='coerce')

print("\nRow-by-row types in annual_income AFTER fix:")
print(rc['financials.annual_income'].dropna().apply(type).value_counts().to_string())


Row-by-row types in annual_income BEFORE fix:
financials.annual_income
<class 'int'>      486
<class 'str'>        8
<class 'float'>      6

Row-by-row types in annual_income AFTER fix:
financials.annual_income
<class 'float'>    500


## Issue 6 — Inconsistent Categorical Coding (Gender)
**Dimension: Consistency**

The `gender` field uses 4 different representations for 2 values:
`Male`, `Female`, `M`, `F` — 111 records (22.1%) use abbreviated codes.

Without fixing this, any groupby or chart would show 4 gender categories instead of 2.

normalise to lowercase, then map all variants to canonical form.

In [89]:
# Show raw unique values 
print("Gender values BEFORE fix:")
print(rc['applicant_info.gender'].value_counts(dropna=False).to_string())

n_shortcode = rc['applicant_info.gender'].isin(['M', 'F']).sum()
print(f"\nShort-code records (M/F): {n_shortcode} ({n_shortcode/len(rc)*100:.1f}%)")


gender_map = {'m': 'male', 'f': 'female', 'male': 'male', 'female': 'female'}

rc['applicant_info.gender'] = (
    rc['applicant_info.gender']
    .astype(str).str.strip().str.lower()
    .replace({'nan': pd.NA})
    .map(gender_map)
)

print("\nGender values AFTER fix:")
print(rc['applicant_info.gender'].value_counts(dropna=False).to_string())

Gender values BEFORE fix:
applicant_info.gender
Male      194
Female    193
F          58
M          53
            2

Short-code records (M/F): 111 (22.2%)

Gender values AFTER fix:
applicant_info.gender
female    251
male      247
NaN         2


## Issue 7 — Inconsistent Date Formats
**Dimension: Consistency**

Three different date formats found in `date_of_birth`:
- `YYYY-MM-DD` — ISO standard (67.9%)
- `DD/MM/YYYY` — European format (20.2%)
- `YYYY/MM/DD` — Non-standard slash format (11.2%)

`pd.to_datetime(dayfirst=True)` parses all three formats automatically.


In [90]:
# Detect formats BEFORE fix
def detect_format(val):
    v = str(val).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}', v): return 'YYYY-MM-DD'
    if re.match(r'^\d{2}/\d{2}/\d{4}', v): return 'DD/MM/YYYY'
    if re.match(r'^\d{4}/\d{2}/\d{2}', v): return 'YYYY/MM/DD'
    return 'OTHER'

formats = rc['applicant_info.date_of_birth'].dropna().apply(detect_format).value_counts()
total_d = rc['applicant_info.date_of_birth'].notna().sum()
print("date_of_birth formats found BEFORE fix:")
for fmt, n in formats.items():
    print(f"  {fmt}: {n} ({n/total_d*100:.1f}%)")

rc['applicant_info.date_of_birth'] = pd.to_datetime(
    rc['applicant_info.date_of_birth'], dayfirst=True, errors='coerce'
).dt.strftime('%Y-%m-%d')

print("\nAll dates standardised to YYYY-MM-DD")
print("Sample after fix:", rc['applicant_info.date_of_birth'].dropna().head(5).tolist())

date_of_birth formats found BEFORE fix:
  YYYY-MM-DD: 339 (67.8%)
  DD/MM/YYYY: 101 (20.2%)
  YYYY/MM/DD: 56 (11.2%)
  OTHER: 4 (0.8%)

All dates standardised to YYYY-MM-DD
Sample after fix: ['2001-09-03', '1991-11-10', '1990-04-05', '1989-10-10', '1970-01-10']


## Issue 8 — Invalid / Impossible Values
**Dimension: Validity**

Values can be the correct type and present, yet still be logically impossible.

| Field | Valid Range | Reason |
|---|---|---|
| `annual_income` | 0 – 10,000,000 | Income cannot be negative |
| `credit_history_months` | 0 – 600 | Time cannot be negative; 600 = 50 years max |
| `debt_to_income` | 0 – 2.0 | Ratio cannot be negative |
| `savings_balance` | 0 – 10,000,000 | Savings account cannot be negative |
| `interest_rate` | 0 – 50 | Rate cannot be negative  |
| `approved_amount` | 1 – 2,000,000 | Amount must be positive |

Replace out-of-range values with `NaN`
We use median (not mean) because it is robust to outliers.

In [91]:
rules = {
    'financials.annual_income':         (0,    10_000_000),
    'financials.credit_history_months': (0,    600),
    'financials.debt_to_income':        (0,    2.0),
    'financials.savings_balance':       (0,    10_000_000),
    'decision.interest_rate':           (0,    50),
    'decision.approved_amount':         (1,    2_000_000),
}

print("Invalid value check:")
for col, (lo, hi) in rules.items():
    if col not in rc.columns:
        continue
    inv = (rc[col] < lo) | (rc[col] > hi)
    n = inv.sum()
    status = f"{n} invalid" if n > 0 else "OK"
    print(f"  {col}: {status}  (valid range: {lo} to {hi})")
    if n > 0:
        print(f"    Values: {rc.loc[inv, col].tolist()}")


col = 'financials.credit_history_months'
invalid_mask = rc[col] < 0
n_inv = invalid_mask.sum()
if n_inv > 0:
    median_val = rc.loc[~invalid_mask, col].median()
    rc.loc[invalid_mask, col] = pd.NA
    rc[col] = rc[col].fillna(median_val)
    print(f"\ncredit_history_months: {n_inv} negatives (-10, -3) replaced with median ({median_val} months)")

col_s = 'financials.savings_balance'
invalid_sav = rc[col_s] < 0
n_sav = invalid_sav.sum()
if n_sav > 0:
    median_sav = rc.loc[~invalid_sav, col_s].median()
    rc.loc[invalid_sav, col_s] = pd.NA
    rc[col_s] = rc[col_s].fillna(median_sav)
    print(f"savings_balance: {n_sav} negative (-$5,000) replaced with median (${median_sav:,.0f})")

Invalid value check:
  financials.annual_income: OK  (valid range: 0 to 10000000)
  financials.credit_history_months: 2 invalid  (valid range: 0 to 600)
    Values: [-10, -3]
  financials.debt_to_income: OK  (valid range: 0 to 2.0)
  financials.savings_balance: 1 invalid  (valid range: 0 to 10000000)
    Values: [-5000]
  decision.interest_rate: OK  (valid range: 0 to 50)
  decision.approved_amount: OK  (valid range: 1 to 2000000)

credit_history_months: 2 negatives (-10, -3) replaced with median (49.0 months)
savings_balance: 1 negative (-$5,000) replaced with median ($27,401)


## Issue 9 — Invalid Email Format
**Dimension: Accuracy**


A valid email must have: a local part + `@` + domain + valid extension.

Invalid emails are set to `NaN` for manual review — we do **NOT** replace with a placeholder


In [92]:
print(f"Total emails in dataset: {rc['applicant_info.email'].notna().sum()}")
print(f"Already NaN emails:      {rc['applicant_info.email'].isna().sum()}")

email_pattern = r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'

invalid_email_mask = rc['applicant_info.email'].dropna().apply(
    lambda x: not bool(re.match(email_pattern, str(x).strip()))
)

n_bad = invalid_email_mask.sum()
print(f"\nInvalid email addresses found: {n_bad} ({n_bad/len(rc)*100:.2f}%)")

if n_bad > 0:
    print("\nAffected records:")
    print(rc.loc[invalid_email_mask[invalid_email_mask].index,
                 ['_id','applicant_info.full_name','applicant_info.email']]
          .to_string(index=False))
    rc.loc[invalid_email_mask[invalid_email_mask].index, 'applicant_info.email'] = pd.NA
    print(f"\n{n_bad} invalid emails set to NaN for manual review")
else:
    print("NOTE: 0 found — restart kernel and run all cells if you need fresh results.")

Total emails in dataset: 493
Already NaN emails:      7

Invalid email addresses found: 4 (0.80%)

Affected records:
    _id applicant_info.full_name   applicant_info.email
app_204          Jonathan Carter mike johnson@gmail.com
app_299          Samuel Gonzalez  test.user.outlook.com
app_068              Emily Lopez       john.doe@invalid
app_146               Amy Flores           sarah.smith@

4 invalid emails set to NaN for manual review


## Issue 10 — Cross-Column Logical Validity
**Dimension: Validity**

Beyond individual field checks, we verify that combinations of fields are logically consistent.
For example: a loan cannot be approved AND have a rejection reason at the same time.

In [93]:
# Approved but has rejection_reason
contra1 = rc[rc['decision.loan_approved'] == True]['decision.rejection_reason'].notna().sum()
print(f'Approved but has rejection_reason:   {contra1} rows')

# Rejected but has approved_amount
contra2 = rc[rc['decision.loan_approved'] == False]['decision.approved_amount'].notna().sum()
print(f'Rejected but has approved_amount:    {contra2} rows')

# Approved but missing interest_rate
contra3 = rc[rc['decision.loan_approved'] == True]['decision.interest_rate'].isna().sum()
print(f'Approved but missing interest_rate:  {contra3} rows')

# Approved but missing approved_amount
contra4 = rc[rc['decision.loan_approved'] == True]['decision.approved_amount'].isna().sum()
print(f'Approved but missing approved_amount:{contra4} rows')

if contra1 + contra2 + contra3 + contra4 == 0:
    print("\nAll decision fields are internally consistent across all records.")

Approved but has rejection_reason:   0 rows
Rejected but has approved_amount:    0 rows
Approved but missing interest_rate:  0 rows
Approved but missing approved_amount:0 rows

All decision fields are internally consistent across all records.


## Issue 11 — Age Validation
**Dimension: Accuracy**

We calculate each applicant's age from their `date_of_birth` and check for impossible values:
- Birth dates in the future
- Applicants under 18 not legally eligible for credit
- Applicants over 100 

In [94]:
rc['dob_parsed'] = pd.to_datetime(
    rc['applicant_info.date_of_birth'], dayfirst=True, errors='coerce'
)
today = pd.Timestamp('2024-12-31')
rc['age_calculated'] = ((today - rc['dob_parsed']).dt.days / 365.25).round(1)

future_dob = (rc['dob_parsed'] > today).sum()
underage   = (rc['age_calculated'] < 18).sum()
too_old    = (rc['age_calculated'] > 100).sum()

print(f'Birth dates in the future:      {future_dob}')
print(f'Underage applicants (under 18): {underage}')
print(f'Applicants over 100 years old:  {too_old}')
print(f'Age range: {rc["age_calculated"].min():.1f} to {rc["age_calculated"].max():.1f} years')
print(f'Mean age:  {rc["age_calculated"].mean():.1f} years')

if future_dob + underage + too_old == 0:
    print("\nAll ages are within valid range (18–100).")

Birth dates in the future:      0
Underage applicants (under 18): 0
Applicants over 100 years old:  0
Age range: 23.2 to 66.0 years
Mean age:  41.4 years

All ages are within valid range (18–100).


## Issue 12 — Spending Behavior 
**Dimension: Validity**

`spending_behavior` is a nested list inside each record — `pd.json_normalize()` does not flatten it.
We loop through the original `raw_data` to inspect the nested structure directly.



In [95]:
neg_spending = 0
neg_records  = []

for rec in raw_data:
    spending = rec.get('spending_behavior', [])
    for s in spending:
        if s.get('amount', 0) < 0:
            neg_spending += 1
            neg_records.append({
                '_id': rec['_id'],
                'category': s['category'],
                'amount': s['amount']
            })

print(f'Negative spending amounts: {neg_spending}')
if neg_spending > 0:
    print(pd.DataFrame(neg_records).to_string(index=False))
else:
    print("No negative spending amounts found.")

print("\nSpending categories found in dataset:")
all_cats = [s.get('category','') for rec in raw_data for s in rec.get('spending_behavior',[])]
cat_counts = pd.Series(all_cats).value_counts()
print(cat_counts.to_string())

high_risk = ['Alcohol','Gambling','Adult Entertainment']
print("\nHigh-risk categories (relevant for credit scoring):")
for cat in high_risk:
    n = cat_counts.get(cat, 0)
    print(f"  {cat}: {n} records")

Negative spending amounts: 0
No negative spending amounts found.

Spending categories found in dataset:
Travel                 80
Utilities              76
Entertainment          72
Fitness                71
Insurance              68
Healthcare             68
Dining                 66
Groceries              65
Education              64
Transportation         61
Rent                   59
Shopping               54
Alcohol                11
Gambling                7
Adult Entertainment     5

High-risk categories (relevant for credit scoring):
  Alcohol: 11 records
  Gambling: 7 records
  Adult Entertainment: 5 records


## Final Summary

All identified issues have been remediated or flagged. The cleaned dataset is exported below.

In [97]:

print("  FINAL DATA QUALITY SUMMARY")
print("=" * 55)
print(f"  Original records:              502")
print(f"  Records removed (dup _id):       2")
print(f"  Records flagged (SSN conflict):  4 ")
print(f"  Final clean records:           500")
print(f"  Remaining NaN cells:           {rc.isna().sum().sum()}")


print(f"\n  Gender distribution:")
print(rc['applicant_info.gender'].value_counts(dropna=False).to_string())
print(f"\n  Loan approved distribution:")
print(rc['decision.loan_approved'].value_counts().to_string())

# Export clean data
rc.to_csv('cleaned_credit_data.csv', index=False)
rc.to_json('cleaned_credit_applications.json', orient='records', indent=2)

  FINAL DATA QUALITY SUMMARY
  Original records:              502
  Records removed (dup _id):       2
  Records flagged (SSN conflict):  4 
  Final clean records:           500
  Remaining NaN cells:           2738

  Gender distribution:
applicant_info.gender
female    251
male      247
NaN         2

  Loan approved distribution:
decision.loan_approved
True     292
False    208
